In [1]:
!pip install -r ../requirements.txt 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 3.5 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 3.5 MB/s eta 0:00:0000:0100:01


In [3]:
import mlflow
import mlflow.pyfunc
from pathlib import Path
import uuid
import logging

class Mira3DPipeline(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.base_path = Path("../data")
        self.base_path.mkdir(exist_ok=True)

    def predict(self, context, model_input):
        try:
            run_uuid = str(uuid.uuid4())
            run_dir = self.base_path / run_uuid
            run_dir.mkdir(parents=True, exist_ok=True)
            log_file = run_dir / "pipeline.log"

            logging.basicConfig(
                filename=log_file,
                filemode="w",
                level=logging.INFO,
                format="%(asctime)s - %(levelname)s - %(message)s"
            )

            logging.info("✅ Frontend request received")
            logging.info(f"Received input: {model_input}")

            return {
                "uuid": run_uuid,
                "ply_path": "DEBUG_SUCCESS"
            }

        except Exception as e:
            print("DEBUG: Pipeline error")
            import traceback
            print(traceback.format_exc())
            return {
                "uuid": "",
                "ply_path": ""
            }


In [4]:
import mlflow
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec

mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment("Mira3D_Pipeline")

input_schema = Schema([
    ColSpec("string", "video_path"),
    ColSpec("boolean", "use_rembg"),
    ColSpec("boolean", "use_downsample"),
])

output_schema = Schema([
    ColSpec("string", "uuid"),
    ColSpec("string", "ply_path"),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="mira3d_pipeline_run") as run:
    mlflow.pyfunc.log_model(
        artifact_path="mira3d_pipeline_v2",
        python_model=Mira3DPipeline(),
        signature=signature,
        artifacts={
            "colmap_bundle": "../export",
            "demo": "../demo"
        },
        pip_requirements="../requirements.txt"
    )

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/mira3d_pipeline",
        name="mira3d_pipeline_v2"
    )


Registered model 'mira3d_pipeline_v2' already exists. Creating a new version of this model...
Created version '6' of model 'mira3d_pipeline_v2'.
